# 🏛️ MEF · Canon y sobrecanon petrolero recaudado por cada municipalidad (Loreto, 2007–hoy)

PERUPETRO publica el canon **por municipalidad** solo en 1993–2002 (Excel de Distribución) y el año en curso (Excel de Canon Municipalidades). Desde 2003 entrega el total municipal a un intermediario (CND hasta 2006; PCM desde 2007), que hace el reparto. Este notebook completa los años que faltan con el **MEF – Consulta Amigable de Ingresos** (`apps5.mineco.gob.pe/transparenciaingresos/mensual/`), monto **recaudado** por cada municipalidad:

**Ruta:** Fuente 5 (Recursos determinados) → Rubro 18 → Tipo de recurso con «PETROLERO» (5: Canon y sobrecanon petrolero y, en años recientes, K: Sub cuenta – Canon y sobrecanon petrolero; se suman) → Nivel de gobierno M → Departamento 16: Loreto → Municipalidad.

- La consulta funciona con *postbacks* de ASP.NET (no se puede armar por URL), así que se maneja un **navegador real** (Playwright) con pausas, como lo haría una persona.
- **2003–2006** no tienen detalle en el MEF (el ingreso figura a nombre del Consejo Nacional de Descentralización).
- **Resultado:** `DATA/PP_canon_municipal_mef_anio.csv` y `CONTROL/CONTROL PP canon_municipal_mef.csv` (suma de municipalidades vs. total de Loreto en el MEF, y vs. el total municipal de PERUPETRO del mismo año: *recaudado* y *transferido* pueden diferir por fechas de registro).
- El mapa (`PP_Mapa_Lotes_Plus.ipynb`) lo usa para los años sin detalle de PERUPETRO y lo rotula con su fuente.

**Bloque 1** instala el navegador (1–2 min). **Bloque 2** descarga y controla.

In [ ]:
#@title Bloque 1 · Instalar el navegador (Playwright + Chromium) { display-mode: "form" }
!pip -q install playwright
!playwright install --with-deps chromium > /dev/null 2>&1 && echo "✅ Chromium listo"


In [ ]:
#@title Bloque 2 · Descargar y controlar { display-mode: "form" }
ANIO_INICIO = 2007  #@param {type:"integer"}
ANIO_FIN = 2026  #@param {type:"integer"}
DEPARTAMENTO = "16: LORETO"  #@param ["16: LORETO", "25: UCAYALI", "20: PIURA", "24: TUMBES", "10: HUANUCO", "08: CUSCO"]
import os, re, time, asyncio, unicodedata, pandas as pd
from datetime import date
DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
BASE = "https://apps5.mineco.gob.pe/transparenciaingresos/mensual"
FUENTE = "MEF, Consulta Amigable de Ingresos (recaudado)"
ruta = "/content/pp_mef"; os.makedirs(ruta, exist_ok=True)

# ── Google Drive ──
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false", fields="nextPageToken, files(id,name)",
                               pageSize=1000, pageToken=token, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out
def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
    if nombre in existentes:
        drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
    else:
        existentes[nombre] = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                                  fields="id", supportsAllDrives=True).execute()["id"]
def bajar(nombre, existentes):
    if nombre not in existentes:
        return None
    p = os.path.join(ruta, nombre)
    with open(p, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=existentes[nombre], supportsAllDrives=True)); hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()
    return pd.read_csv(p, encoding="utf-8-sig")
datos_drive, control_drive = archivos_en(DRIVE_FOLDER_ID), archivos_en(DRIVE_FOLDER_CONTROL_ID)

# ── Lector: canon y sobrecanon petrolero recaudado por cada municipalidad (MEF, Consulta Amigable de Ingresos) ──
# La consulta es ASP.NET con postbacks (Navegar_3.aspx?y=AAAA + __doPostBack): no se arma por URL como la de gasto,
# así que se maneja un navegador real (Playwright): elegir la fila (radio) y pulsar el botón de la dimensión siguiente.
import re, asyncio

DIMS = ("Fuente de Financiamiento", "Rubro", "Tipo de Recurso", "Nivel de Gobierno", "Gob.Loc./Mancom.", "Sector", "Pliego",
        "Unidad Ejecutora", "Departamento", "Provincia", "Municipalidad", "Tipo de Ingreso")

JS_ESTADO = r"""() => {
  const limpio = s => (s || '').replace(/\s+/g, ' ').trim();
  const num = s => { const t = limpio(s).replace(/,/g, ''); return /^-?\d+(\.\d+)?$/.test(t) ? parseFloat(t) : null; };
  const out = {dim: null, filas: [], migas: [], botones: []};
  const radios = [...document.querySelectorAll('input[type=radio]')];
  for (const tr of document.querySelectorAll('tr')) {
    const celdas = [...tr.children].filter(x => x.tagName === 'TD' || x.tagName === 'TH');
    const tx = celdas.map(c => limpio(c.innerText));
    if (!out.dim && tx.some(t => /^Recaudado/.test(t)) && !tr.querySelector('input[type=radio]'))
      out.dim = tx.find(t => t && !/^(PIA|PIM|Recaudado)/.test(t)) || null;
    const r = tr.querySelector('input[type=radio]');
    const etiqueta = tx.find(t => /^[0-9A-ZÑ]{1,3}(-\d+)?:\s/.test(t));
    const nums = tx.map(num).filter(v => v !== null);
    if (r && etiqueta) out.filas.push({i: radios.indexOf(r), etiqueta, recaudado: nums.length ? nums[nums.length - 1] : null});
    else if (!r && etiqueta === undefined && tx.length) {
      const m = tx.find(t => /^(TOTAL|Fuente de Financiamiento|Rubro|Tipo de Recurso|Nivel de Gobierno|Gob\.Loc|Sector|Pliego|Departamento|Provincia)\b/.test(t));
      if (m && nums.length) out.migas.push({etiqueta: m, recaudado: nums[nums.length - 1]});
    }
  }
  out.botones = [...document.querySelectorAll('input[type=submit],input[type=button],button')].map(b => limpio(b.value || b.innerText)).filter(x => x);
  return out;
}"""

class MEFIngresos:
    def __init__(self, page, base, log=print, pausa=1.0):
        self.page, self.base, self.log, self.pausa = page, base.rstrip("/"), log, pausa
        self.frame = page.main_frame

    async def _esperar_recarga(self, accion, timeout=60000):
        await self.frame.evaluate("() => { window.__pp = 1; }")
        await accion()
        try:
            await self.frame.wait_for_function("() => window.__pp === undefined", timeout=timeout)
        except Exception:
            pass                                          # sin recarga (selección solo en el navegador)
        await self.frame.wait_for_load_state("domcontentloaded")
        await asyncio.sleep(self.pausa)

    async def estado(self):
        return await self.frame.evaluate(JS_ESTADO)

    async def abrir_anio(self, anio):
        for url in (f"{self.base}/Navegar_3.aspx?y={anio}", f"{self.base}/default.aspx?y={anio}"):
            await self.page.goto(url, wait_until="domcontentloaded", timeout=90000)
            await asyncio.sleep(self.pausa)
            fr = next((f for f in self.page.frames if "navegar" in f.url.lower()), self.page.main_frame)
            self.frame = fr
            e = await self.estado()
            if e["filas"]:
                return e
        raise RuntimeError(f"{anio}: la consulta no mostró filas (¿bloqueo o cambio de la página?)")

    async def elegir(self, patron):
        e = await self.estado()
        f = next((x for x in e["filas"] if re.search(patron, x["etiqueta"], re.I)), None)
        if f is None:
            return None
        radio = self.frame.locator("input[type=radio]").nth(f["i"])
        await self._esperar_recarga(lambda: radio.click(), timeout=1500)
        return f

    async def ir_a(self, dimension):
        e = await self.estado()
        if e["dim"] and e["dim"].lower().startswith(dimension.lower()):
            return True
        if dimension not in e["botones"]:
            return False
        b = self.frame.locator(f"input[value='{dimension}'], button:has-text('{dimension}')").first
        await self._esperar_recarga(lambda: b.click())
        e = await self.estado()
        return bool(e["dim"] and e["dim"].lower().startswith(dimension.lower()))

    async def municipalidades(self, anio, tipo_patron, dep_patron=r"^16:\s*LORETO"):
        """Filas «Municipalidad» del departamento para un tipo de recurso. Devuelve (filas, total_departamento, ruta)."""
        await self.abrir_anio(anio)
        ruta = []
        pasos = [(r"^5:\s*RECURSOS DETERMINADOS", "Rubro"), (r"^18:", "Tipo de Recurso"), (tipo_patron, "Nivel de Gobierno"),
                 (r"^M:", None), (dep_patron, "Municipalidad")]
        for patron, siguiente in pasos:
            f = await self.elegir(patron)
            if f is None:
                e = await self.estado()
                return None, None, ruta + [f"NO ENCONTRADO {patron} en «{e['dim']}»"]
            ruta.append(f["etiqueta"])
            if siguiente is None:                          # nivel M: en años recientes aparece «Gob.Loc./Mancom.» antes del departamento
                if await self.ir_a("Gob.Loc./Mancom."):
                    g = await self.elegir(r"^M:")
                    if g: ruta.append(g["etiqueta"])
                if not await self.ir_a("Departamento"):
                    return None, None, ruta + ["SIN BOTÓN Departamento"]
                continue
            if not await self.ir_a(siguiente):
                return None, None, ruta + [f"SIN BOTÓN {siguiente}"]
        e = await self.estado()
        total = next((m["recaudado"] for m in e["migas"] if m["etiqueta"].startswith("Departamento")), None)
        return e["filas"], total, ruta

    async def tipos_petroleros(self, anio):
        """Tipos de recurso del rubro 18 que dicen «PETROLERO» (5 y K, según el año)."""
        await self.abrir_anio(anio)
        for patron, siguiente in ((r"^5:\s*RECURSOS DETERMINADOS", "Rubro"), (r"^18:", "Tipo de Recurso")):
            if await self.elegir(patron) is None or not await self.ir_a(siguiente):
                return []
        e = await self.estado()
        return [(f["etiqueta"], f["recaudado"]) for f in e["filas"] if re.search("PETROLER", f["etiqueta"], re.I)]

def nombre_municipalidad(etiqueta):
    """«01-301422: MUNICIPALIDAD PROVINCIAL DE MAYNAS - IQUITOS» → (código, nivel, municipalidad, provincia)."""
    cod, _, txt = etiqueta.partition(":")
    txt = txt.strip().upper()
    m = re.match(r"MUNICIPALIDAD\s+(PROVINCIAL|DISTRITAL)\s+(?:(?:DE\s+LA|DEL|DE)\s+)?(.*)$", txt)
    if not m:
        return cod.strip(), "", txt, ""
    nivel, nom = m.group(1), m.group(2).strip()
    if nivel == "PROVINCIAL":
        prov, _, capital = nom.partition(" - ")
        return cod.strip(), nivel, (capital or prov).strip(), prov.strip()
    return cod.strip(), nivel, nom, ""


def norm(t):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()).strip().upper()
DEP = DEPARTAMENTO.split(":", 1)[1].strip()
# referencias de PERUPETRO (si existen en DATA): total municipal por año y provincia de cada municipalidad distrital
dist_pp = bajar("PP_canon_distribucion_mes.csv", datos_drive)
mun_pp = bajar("PP_canon_municipal_mes.csv", datos_drive)
pp_mun = (dist_pp[(dist_pp["DEPARTAMENTO"].map(norm) == DEP) & (dist_pp["TIPO_BENEFICIARIO"] == "MUNICIPALIDADES")]
          .groupby("AÑO")["MONTO_SOLES"].sum().to_dict() if dist_pp is not None else {})
prov_de = ({norm(m): norm(p) for m, p in zip(mun_pp["MUNICIPALIDAD"], mun_pp["PROVINCIA"]) if isinstance(p, str)}
           if mun_pp is not None else {})

from playwright.async_api import async_playwright
filas, ctrl = [], []
async def correr():
    async with async_playwright() as p:
        nav = await p.chromium.launch()
        pg = await (await nav.new_context(locale="es-PE", user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                                                                       "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")).new_page()
        m = MEFIngresos(pg, BASE, pausa=1.5)
        for anio in range(ANIO_INICIO, ANIO_FIN + 1):
            for intento in (1, 2):
                try:
                    tipos = await m.tipos_petroleros(anio)
                    if not tipos:
                        ctrl.append({"AÑO": anio, "ESTADO": "SIN TIPO DE RECURSO PETROLERO EN EL RUBRO 18"}); break
                    suma = tot = 0.0; n = 0; detalle = []
                    for etq, rec_nac in tipos:
                        cod_t = etq.split(":")[0].strip()
                        fs, total, recorrido = await m.municipalidades(anio, "^" + re.escape(cod_t) + r":\s", r"^" + re.escape(DEPARTAMENTO.split(":")[0]) + r":")
                        if fs is None:
                            detalle.append(f"{etq}: {' > '.join(recorrido)}"); continue
                        for f in fs:
                            if not re.search("MUNICIPALIDAD", f["etiqueta"], re.I):
                                continue
                            cod, nivel, mu, prov = nombre_municipalidad(f["etiqueta"])
                            filas.append({"AÑO": anio, "DEPARTAMENTO": DEP, "CODIGO": cod, "MUNICIPALIDAD_MEF": f["etiqueta"].split(":", 1)[1].strip(),
                                          "MUNICIPALIDAD": mu, "NIVEL": nivel, "PROVINCIA": prov or prov_de.get(norm(mu), ""),
                                          "TIPO_RECURSO": etq, "RECAUDADO_SOLES": f["recaudado"] or 0.0, "FUENTE": FUENTE})
                            suma += f["recaudado"] or 0.0; n += 1
                        tot += total or 0.0
                        detalle.append(f"{etq}: S/ {total or 0:,.0f}")
                    ref = pp_mun.get(anio)
                    ctrl.append({"AÑO": anio, "TIPOS": " | ".join(detalle), "MUNICIPALIDADES": n, "SUMA_MUNICIPALIDADES": round(suma, 2),
                                 "TOTAL_DEPARTAMENTO_MEF": round(tot, 2), "DIF": round(suma - tot, 2),
                                 "PERUPETRO_MUNICIPAL": round(ref, 2) if ref else None,
                                 "MEF_SOBRE_PERUPETRO": round(suma / ref, 4) if ref else None,
                                 "ESTADO": ("OK" if n and abs(suma - tot) <= 1 else "REVISAR" if n else "SIN MUNICIPALIDADES")})
                    print(f"✅ {anio}: {n} municipalidades · S/ {suma/1e6:,.2f} millones" + (f" · PERUPETRO S/ {ref/1e6:,.2f} millones" if ref else ""))
                    break
                except Exception as e:
                    print(f"⚠️ {anio} intento {intento}: {type(e).__name__}: {str(e)[:160]}")
                    try:
                        png = os.path.join(ruta, f"CONTROL PP mef_error_{anio}.png"); await pg.screenshot(path=png, full_page=True)
                        subir(png, DRIVE_FOLDER_CONTROL_ID, control_drive, mime="image/png")
                    except Exception:
                        pass
                    if intento == 2:
                        ctrl.append({"AÑO": anio, "ESTADO": f"ERROR: {type(e).__name__}: {str(e)[:150]}"})
                    await asyncio.sleep(10)
        await nav.close()
await correr()

# ── Guardar (se acumula: cada corrida reemplaza los años que vuelve a leer) ──
nuevo = pd.DataFrame(filas)
previo = bajar("PP_canon_municipal_mef_anio.csv", datos_drive)
if previo is not None and len(nuevo):
    previo = previo[~((previo["DEPARTAMENTO"] == DEP) & previo["AÑO"].isin(set(nuevo["AÑO"])))]
    nuevo = pd.concat([previo, nuevo], ignore_index=True)
elif previo is not None:
    nuevo = previo
if len(nuevo):
    nuevo = nuevo.sort_values(["DEPARTAMENTO", "AÑO", "NIVEL", "MUNICIPALIDAD", "TIPO_RECURSO"], ascending=[True, True, False, True, True])
    p = os.path.join(ruta, "PP_canon_municipal_mef_anio.csv"); nuevo.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, DRIVE_FOLDER_ID, datos_drive)
c = pd.DataFrame(ctrl); c.insert(0, "DEPARTAMENTO", DEP); c.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
previo_c = bajar("CONTROL PP canon_municipal_mef.csv", control_drive)
if previo_c is not None:
    c = pd.concat([previo_c[~((previo_c["DEPARTAMENTO"] == DEP) & previo_c["AÑO"].isin(set(c["AÑO"])))], c], ignore_index=True).sort_values(["DEPARTAMENTO", "AÑO"])
p = os.path.join(ruta, "CONTROL PP canon_municipal_mef.csv"); c.to_csv(p, index=False, encoding="utf-8-sig")
subir(p, DRIVE_FOLDER_CONTROL_ID, control_drive)
print(f"\n☁️ DATA/PP_canon_municipal_mef_anio.csv ({len(nuevo)} filas) · CONTROL/CONTROL PP canon_municipal_mef.csv")
display(c[c["DEPARTAMENTO"] == DEP].drop(columns=["FECHA_EXTRACCION"]))
